In [ ]:
import json
import re
import tempfile
import zipfile
from pathlib import Path

import h5py
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

IMG_SIZE = (224, 224)

print(f"TensorFlow version: {tf.__version__}")
if not tf.__version__.startswith("2.14."):
    raise RuntimeError(f"Use the TensorFlow 2.14 venv/kernel for conversion, got {tf.__version__}")


def natural_key(name):
    match = re.match(r"^(.*?)(?:_(\d+))?$", name)
    return match.group(1), int(match.group(2)) if match.group(2) else -1


def archive_config(keras_path):
    with zipfile.ZipFile(keras_path) as archive:
        return json.loads(archive.read("config.json"))


def num_classes_from_config(config):
    dense_layers = [
        layer for layer in config["config"]["layers"]
        if layer["class_name"] == "Dense"
    ]
    if len(dense_layers) != 1:
        raise RuntimeError(f"Expected one Dense output layer, found {len(dense_layers)}")
    return dense_layers[0]["config"]["units"]


def build_tf214_model(num_classes):
    tf.keras.backend.clear_session()

    data_augmentation = tf.keras.Sequential([
        layers.RandomTranslation(height_factor=0.05, width_factor=0.05, fill_mode="constant", fill_value=0.0),
        layers.RandomRotation(factor=0.014, fill_mode="constant", fill_value=0.0),
        layers.RandomBrightness(factor=0.2),
        layers.RandomContrast(factor=0.2),
        layers.GaussianNoise(stddev=0.05),
    ])
    base_model = EfficientNetB0(weights="imagenet", include_top=False, input_shape=IMG_SIZE + (3,))

    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs)


def group_arrays(h5_file, group_path):
    group = h5_file[group_path]
    if "vars" not in group:
        return []
    variables = group["vars"]
    return [variables[key][()] for key in sorted(variables.keys(), key=int)]


def assign_weights(layer, arrays, source_name):
    expected = [tuple(weight.shape.as_list()) for weight in layer.weights]
    actual = [array.shape for array in arrays]
    if expected != actual:
        raise RuntimeError(f"Shape mismatch for {layer.name} <- {source_name}: expected {expected}, got {actual}")
    layer.set_weights(arrays)


def assign_by_class_order(tf_layers, h5_parent, layer_class, group_prefix):
    target_layers = [layer for layer in tf_layers if isinstance(layer, layer_class) and layer.weights]
    source_groups = sorted(
        [name for name in h5_parent.keys() if name == group_prefix or name.startswith(group_prefix + "_")],
        key=natural_key,
    )
    if len(target_layers) != len(source_groups):
        raise RuntimeError(
            f"Layer count mismatch for {layer_class.__name__}: "
            f"expected {len(target_layers)}, got {len(source_groups)}"
        )

    for layer, group_name in zip(target_layers, source_groups):
        assign_weights(layer, group_arrays(h5_parent.file, f"{h5_parent.name}/{group_name}"), group_name)


def first_layer_of_type(layers_to_search, layer_class):
    matches = [layer for layer in layers_to_search if isinstance(layer, layer_class)]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {layer_class.__name__} layer, found {len(matches)}")
    return matches[0]


def first_nested_model_containing(layers_to_search, layer_class):
    matches = [
        layer for layer in layers_to_search
        if isinstance(layer, tf.keras.Model)
        and any(isinstance(inner_layer, layer_class) for inner_layer in layer.layers)
    ]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one nested model containing {layer_class.__name__}, found {len(matches)}")
    return matches[0]


def load_keras3_archive_weights(model, keras_path):
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_dir = Path(temp_dir)
        with zipfile.ZipFile(keras_path) as archive:
            archive.extract("model.weights.h5", temp_dir)

        with h5py.File(temp_dir / "model.weights.h5", "r") as h5_file:
            assign_weights(model.layers[-1], group_arrays(h5_file, "layers/dense"), "layers/dense")

            base_model = first_nested_model_containing(model.layers, tf.keras.layers.Conv2D)
            h5_layers = h5_file["layers/functional/layers"]
            assign_weights(
                first_layer_of_type(base_model.layers, tf.keras.layers.Normalization),
                group_arrays(h5_file, "layers/functional/layers/normalization"),
                "normalization",
            )
            assign_by_class_order(base_model.layers, h5_layers, tf.keras.layers.Conv2D, "conv2d")
            assign_by_class_order(base_model.layers, h5_layers, tf.keras.layers.DepthwiseConv2D, "depthwise_conv2d")
            assign_by_class_order(base_model.layers, h5_layers, tf.keras.layers.BatchNormalization, "batch_normalization")


def convert_archive(keras_path, tflite_path):
    keras_path = Path(keras_path)
    tflite_path = Path(tflite_path)
    if not keras_path.exists():
        raise FileNotFoundError(f"Missing model file: {keras_path.resolve()}")

    metadata = json.loads(zipfile.ZipFile(keras_path).read("metadata.json"))
    config = archive_config(keras_path)
    num_classes = num_classes_from_config(config)

    print(f"Loading {keras_path} saved by Keras {metadata.get('keras_version')} with {num_classes} classes")
    model = build_tf214_model(num_classes)
    load_keras3_archive_weights(model, keras_path)

    print(f"Converting {keras_path} -> {tflite_path}")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()

    tflite_path.write_bytes(tflite_model)
    print(f"Saved {tflite_path} ({len(tflite_model):,} bytes)")


for keras_path, tflite_path in [
    ("palm_recognition.keras", "palm_recognition.tflite"),
    ("palm_recognition_lbp.keras", "palm_recognition_lbp.tflite"),
]:
    convert_archive(keras_path, tflite_path)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def prepare_tflite_input(values, input_detail):
    dtype = input_detail["dtype"]
    if np.issubdtype(dtype, np.integer):
        scale, zero_point = input_detail["quantization"]
        if scale > 0:
            values = values / scale + zero_point
        limits = np.iinfo(dtype)
        values = np.clip(values, limits.min, limits.max)
    return values.astype(dtype)


def dequantize_tflite_output(values, output_detail):
    scale, zero_point = output_detail["quantization"]
    if np.issubdtype(values.dtype, np.integer) and scale > 0:
        return (values.astype(np.float32) - zero_point) * scale
    return values


def predict_tflite_dataset(model_path, dataset):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    input_detail = interpreter.get_input_details()[0]
    interpreter.resize_tensor_input(input_detail["index"], [1, IMG_SIZE[0], IMG_SIZE[1], 3], strict=False)
    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    y_true = []
    y_pred = []

    for images, labels in dataset:
        images = images.numpy()
        labels = labels.numpy().astype(int)

        for image, label in zip(images, labels):
            sample = prepare_tflite_input(image[np.newaxis, ...], input_detail)
            interpreter.set_tensor(input_detail["index"], sample)
            interpreter.invoke()
            prediction = dequantize_tflite_output(interpreter.get_tensor(output_detail["index"]), output_detail)

            y_true.append(label)
            y_pred.append(int(np.argmax(prediction[0])))

    return np.array(y_true), np.array(y_pred)


def evaluate_tflite_model(title, model_path, test_dir):
    model_path = Path(model_path)
    test_dir = Path(test_dir)

    if not model_path.exists():
        raise FileNotFoundError(f"Missing TFLite model: {model_path.resolve()}")
    if not test_dir.exists():
        raise FileNotFoundError(f"Missing test dataset directory: {test_dir.resolve()}")

    print("=" * 60)
    print(title)
    print("=" * 60)

    test_ds = tf.keras.preprocessing.image_dataset_from_directory(
        test_dir,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
        shuffle=False,
    )

    class_names = test_ds.class_names
    labels = list(range(len(class_names)))
    y_true, y_pred = predict_tflite_dataset(model_path, test_ds)
    accuracy = accuracy_score(y_true, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(12, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=90)
    plt.title(f"Confusion Matrix - {title}")
    plt.tight_layout()
    plt.show()

    return {
        "accuracy": accuracy,
        "y_true": y_true,
        "y_pred": y_pred,
        "class_names": class_names,
        "confusion_matrix": cm,
    }


tflite_results = {}
for title, model_path, test_dir in [
    ("TFLite CNN", "palm_recognition.tflite", "Dataset/dataset_preprocessing/test"),
    ("TFLite CNN + LBP", "palm_recognition_lbp.tflite", "Dataset/dataset_lbp/test"),
]:
    tflite_results[title] = evaluate_tflite_model(title, model_path, test_dir)

print("=" * 60)
print("TFLITE ACCURACY SUMMARY")
print("=" * 60)
for title, result in tflite_results.items():
    print(f"{title}: {result['accuracy']:.4f}")

TFLite CNN
Found 166 files belonging to 28 classes.


KeyboardInterrupt: 